In [18]:
import torch.optim as optim

from src.load_and_save import save_model
from src.training import get_accuracy
from src.utils import device
from settings import settings
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import torch.nn as nn
import torch

In [19]:
# Define the number of classes in MNIST (digits 0-9)
num_classes = 10

# Define the transformation to apply to the images
transform = transforms.Compose([
    transforms.ToTensor(),
])

# Custom transform to one-hot encode the labels
class OneHotEncode:
    def __init__(self, num_classes):
        self.num_classes = num_classes

    def __call__(self, label):
        return torch.eye(self.num_classes)[label]

# Load the full training dataset
full_train_dataset = datasets.MNIST(
    root=settings.data_path,
    train=True,
    download=True,
    transform=transform,
    target_transform=OneHotEncode(num_classes)
)

# Split the full training dataset into training and validation datasets
train_size = int(0.8 * len(full_train_dataset))  # 80% for training
val_size = len(full_train_dataset) - train_size  # 20% for validation
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

# Load the test dataset
test_dataset = datasets.MNIST(
    root=settings.data_path,
    train=False,
    download=True,
    transform=transform,
    target_transform=OneHotEncode(num_classes)
)

# Create DataLoaders for training, validation, and test sets
train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=512, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=2048, shuffle=False)

In [20]:
from src.improved_model import BinarizingCNN

# Instantiate the model
model = BinarizingCNN().to(device)
model.set_scramble_distance(0.05)
#criterion = nn.CrossEntropyLoss()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=float(1e-2))

In [21]:
from src.training import get_average_separation

# Training loop
num_epochs: int = 500
target_accuracy: float = .95
maximum_scramble_distance: float = 5.0

test_data, _ = next(iter(test_dataloader))
test_data.to(device)


for epoch in range(num_epochs):
    for inputs, labels in train_dataloader:
        if epoch == 0:
            break
        # Forward pass
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        model.layer2.scale.data.clamp_(min=1.0)
        model.layer3.scale.data.clamp_(min=1.0)


    # Assess progress:
    # data: np.ndarray = get_intermediate_outputs_as_numpy(model, train_dataloader)
    validation_accuracy: float = get_accuracy(model, val_dataloader)
    if validation_accuracy > target_accuracy:
        model.set_scramble_distance(min(model.scramble_distance + .1, maximum_scramble_distance))
        print(f"Scramble distance: {model.scramble_distance:.2f}")
    print(f'Epoch [{epoch+1}/{num_epochs}], Accuracy: {validation_accuracy:.4f}')


Epoch [1/500], Accuracy: 0.0940
Epoch [2/500], Accuracy: 0.9167
Epoch [3/500], Accuracy: 0.9406
Scramble distance: 0.15
Epoch [4/500], Accuracy: 0.9502
Epoch [5/500], Accuracy: 0.9489
Scramble distance: 0.25
Epoch [6/500], Accuracy: 0.9529
Epoch [7/500], Accuracy: 0.9441
Epoch [8/500], Accuracy: 0.9467
Scramble distance: 0.35
Epoch [9/500], Accuracy: 0.9508
Epoch [10/500], Accuracy: 0.9458
Epoch [11/500], Accuracy: 0.9464
Epoch [12/500], Accuracy: 0.9479
Scramble distance: 0.45
Epoch [13/500], Accuracy: 0.9523
Epoch [14/500], Accuracy: 0.9448
Epoch [15/500], Accuracy: 0.9474
Epoch [16/500], Accuracy: 0.9477
Epoch [17/500], Accuracy: 0.9482
Scramble distance: 0.55
Epoch [18/500], Accuracy: 0.9523
Epoch [19/500], Accuracy: 0.9480
Epoch [20/500], Accuracy: 0.9486
Epoch [21/500], Accuracy: 0.9499
Scramble distance: 0.65
Epoch [22/500], Accuracy: 0.9503
Epoch [23/500], Accuracy: 0.9457
Epoch [24/500], Accuracy: 0.9472
Epoch [25/500], Accuracy: 0.9476
Epoch [26/500], Accuracy: 0.9488
Epoch [

KeyboardInterrupt: 

In [22]:
from src.improved_model import BinarizingNetwork


def get_signed_accuracy(model: BinarizingNetwork, dataloader: DataLoader) -> float:
    model.eval_mode()

    with torch.no_grad():  # Disable gradient computation
        all_correct: int = 0
        for inputs, labels in dataloader:
            # Move inputs and labels to the specified device
            inputs, labels = inputs.to(device), labels.to(device)
            outputs: torch.Tensor = model(inputs)
            comparison: torch.Tensor = torch.argmax(outputs, axis=1) == torch.argmax(
                labels, axis=1
            )
            all_correct += sum(comparison)
        accuracy: float = all_correct / len(dataloader.dataset)
    model.train_mode()
    return accuracy
test_data = test_data.to(device)
model.eval_mode()
model(test_data[0:1])

d = test_data[0:1]
#print(model.float_to_binary_layer(d))

print(model.second_layer(model.float_to_binary_layer(d)))
#
# # model(test_data[0,...])
print(get_signed_accuracy(model, val_dataloader))


tensor([[-1.,  1.,  1., -1.,  1., -1., -1., -1., -1., -1.,  1., -1., -1., -1.,
          1., -1., -1.,  1., -1., -1.,  1.,  1.,  1., -1.,  1., -1.,  1., -1.,
         -1.,  1.,  1., -1.,  1.,  1.,  1., -1., -1.,  1., -1., -1.,  1., -1.,
         -1.,  1.,  1.,  1.,  1.,  1., -1.,  1., -1.,  1., -1., -1.,  1., -1.,
         -1., -1.,  1., -1., -1.,  1.,  1., -1., -1., -1.,  1., -1.,  1.,  1.,
          1., -1.,  1.,  1., -1.,  1.,  1.]], device='cuda:0',
       grad_fn=<SignBackward0>)
tensor(0.9126, device='cuda:0')


In [ ]:
from torch import softmax

model.eval_mode()
model.layer3.scale
#1 + softmax(model.layer2.bias, dim=0)

In [ ]:
model.layer3